# One-Class SVM - multi-run experiments

In [ ]:
import sys
import time
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score

sys.path.append("../src")
from data import make_optuna_subsample, make_final_subsample
from optuna_utils import run_study
from metrics import find_best_f1_threshold, minmax_scale_scores, evaluate_scores, print_metrics
from results import build_experiment_record, save_record_json, get_memory_mb

In [ ]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["CICIDS", "UNSW_NB15"]
SEED = 29
N_TRIALS = 200

DATASET_VERSION = "v1"
PREPROCESSING_VERSION = "v1"
SPLIT_METHOD = "stratified_train_val_test_fixed_seed"
MODEL_TYPE = "OneClassSVM"
FUSION_STRATEGY = "none"

RUN_CONFIGS = [
    dict(run_index=1, n_train_opt=7000,  n_val_opt=3000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run1_small_opt_sample"),
    dict(run_index=2, n_train_opt=21000, n_val_opt=9000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run2_medium_opt_sample"),
    dict(run_index=3, n_train_opt=35000, n_val_opt=15000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run3_large_opt_sample"),
]

# ---------------------------------------------------------------------------
# NOTE on OneClassSVM:
# OneClassSVM natively supports .fit() on training data and .score_samples()
# / .decision_function() on unseen points, so no scoring workaround is
# needed (unlike DBSCAN). Higher decision_function value = more "normal";
# we negate it so that, consistent with every other model in this suite,
# a HIGHER score means MORE anomalous.
# Training cost scales roughly quadratically with n_samples for the RBF
# kernel, so the optuna search subsamples stay small and the final
# subsample train size may need capping on very large datasets.
# ---------------------------------------------------------------------------


In [ ]:
def make_objective(train_x, train_y, val_x, val_y):
    def objective(trial):
        n_pos = int(val_y.sum())
        n_neg = len(val_y) - n_pos
        if n_pos < 5 or n_neg < 5:
            raise optuna.exceptions.TrialPruned()

        kernel = trial.suggest_categorical("kernel", ["rbf", "poly", "sigmoid"])
        nu = trial.suggest_float("nu", 0.01, 0.5)
        gamma = trial.suggest_categorical("gamma", ["scale", "auto"])
        degree = trial.suggest_int("degree", 2, 5) if kernel == "poly" else 3
        coef0 = trial.suggest_float("coef0", 0.0, 1.0) if kernel in ("poly", "sigmoid") else 0.0

        try:
            clf = OneClassSVM(
                kernel=kernel,
                nu=nu,
                gamma=gamma,
                degree=degree,
                coef0=coef0,
            )
            clf.fit(train_x)
            val_scores = -clf.decision_function(val_x)
            auc = roc_auc_score(val_y, val_scores)
        except ValueError:
            raise optuna.exceptions.TrialPruned()

        return auc
    return objective


def fit_and_score_ocsvm(params, train_x, train_y, val_x, test_x):
    clf = OneClassSVM(
        kernel=params["kernel"],
        nu=params["nu"],
        gamma=params["gamma"],
        degree=params.get("degree", 3),
        coef0=params.get("coef0", 0.0),
    )

    gc.collect()
    mem_before = get_memory_mb()

    start_train = time.time()
    clf.fit(train_x)
    runtime_train = time.time() - start_train
    mem_after_train = get_memory_mb()

    val_scores_raw = -clf.decision_function(val_x)
    scores_val = minmax_scale_scores(val_scores_raw)

    start_inference = time.time()
    test_scores_raw = -clf.decision_function(test_x)
    runtime_inference = time.time() - start_inference
    mem_after_inference = get_memory_mb()

    scores_test = minmax_scale_scores(test_scores_raw)
    memory_peak = max(mem_before, mem_after_train, mem_after_inference)

    return clf, scores_val, scores_test, runtime_train, runtime_inference, memory_peak


def run_experiment(dataset, run_cfg):
    run_index = run_cfg["run_index"]
    study_name = f"OneClassSVM_{dataset}_run{run_index}"

    train_x, train_y, val_x, val_y = make_optuna_subsample(
        dataset, SEED, run_cfg["n_train_opt"], run_cfg["n_val_opt"]
    )
    objective = make_objective(train_x, train_y, val_x, val_y)
    study = run_study(objective, study_name, SEED, N_TRIALS, results_dir=RESULTS_DIR)

    train_x, train_y, val_x, val_y, test_x, test_y = make_final_subsample(
        dataset, SEED,
        run_cfg["n_train_final"], run_cfg["n_val_final"], run_cfg["n_test_final"],
    )

    clf, scores_val, scores_test, runtime_train, runtime_inference, memory_peak = fit_and_score_ocsvm(
        study.best_params, train_x, train_y, val_x, test_x
    )

    best_threshold, best_f1_val, best_prec_val, best_rec_val = find_best_f1_threshold(val_y, scores_val)
    print(f"[{dataset} run{run_index}] threshold={best_threshold:.4f} "
          f"F1={best_f1_val:.4f} P={best_prec_val:.4f} R={best_rec_val:.4f}")

    metrics = evaluate_scores(test_y, scores_test, threshold=best_threshold)
    print_metrics(f"OneClassSVM final - {dataset} run{run_index}", metrics)

    model_dir = MODELS_DIR / MODEL_TYPE
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / f"{dataset}_run{run_index}.joblib"
    joblib.dump(clf, model_path)

    record = build_experiment_record(
        dataset_name=dataset,
        dataset_version=DATASET_VERSION,
        split_method=SPLIT_METHOD,
        seed=SEED,
        preprocessing_version=PREPROCESSING_VERSION,
        model_type=MODEL_TYPE,
        fusion_strategy=FUSION_STRATEGY,
        hyperparameters=study.best_params,
        threshold=best_threshold,
        scores_test=scores_test,
        test_y=test_y,
        runtime_train=runtime_train,
        runtime_inference=runtime_inference,
        memory_peak=memory_peak,
        notes=run_cfg["notes"],
    )
    save_record_json(record, RESULTS_DIR, run_index, MODEL_TYPE, dataset)
    return record

In [ ]:
all_records = []

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[0])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[0])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[1])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[1])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[2])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[2])
all_records.append(rec)
pd.DataFrame(all_records)